# LangChain Demo: Pitti Fashion Exhibitor Analysis

This notebook demonstrates key LangChain capabilities:
1. **Text Loading & Splitting** - Prepare data for embeddings
2. **Vector Embeddings** - Convert text to numerical vectors
3. **Semantic Search** - Find similar items by meaning
4. **RAG (Retrieval-Augmented Generation)** - Answer questions with context
5. **Chain Composition** - Build complex workflows

## Step 1: Load Data

In [12]:
import pandas as pd
import numpy as np
import os

# Load Pitti exhibitor data
df = pd.read_excel('pitti.xlsx')
print(f"Loaded {len(df)} exhibitors")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample:")
print(df.head(2))

Loaded 797 exhibitors

Columns: ['description', 'tag', 'link']

Sample:
                                         description  \
0  The company was founded in 1972 when Goliardo ...   
1  A. Leyva S.A. was founded in the early 1960's ...   

                                                 tag  \
0  leather wear, casual jackets, artisan attentio...   
1  Play video, leather wear, braces, belts, monk,...   

                                                link  
0  /en/e-pitti/fairs/pittiuomo97/exhibitors/A/a-f...  
1  /en/e-pitti/fairs/pittiuomo97/exhibitors/A/a--...  


## Step 2: Explore Data

In [13]:
# Data quality
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal tags: {df['tag'].nunique()}")
print(f"\nTop categories:")
print(df['tag'].value_counts().head(10))

Missing values:
description    28
tag             2
link            0
dtype: int64

Total tags: 667

Top categories:
tag
artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual                         18
shirts, artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual                  9
knitwear, artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual                7
denim, trousers, artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual         7
globetrotter, high performance, urban fashion, gymwear, sport heritage, sportswear, sporty chic                                                                     7
trousers, artisan attention, contemporary Classic

## Step 3: Text Splitting (LangChain Fundamentals)

Before creating embeddings, we need to split long texts into manageable chunks.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create documents from descriptions
documents = []
for idx, row in df[df['description'].notna()].iterrows():
    doc_text = f"Company: {row['tag']}\n\nDescription: {row['description']}"
    documents.append(doc_text)

print(f"Total documents: {len(documents)}")
print(f"Sample document:\n{documents[0][:300]}...")

Total documents: 769
Sample document:
Company: leather wear, casual jackets, artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual

Description: The company was founded in 1972 when Goliardo and Franco Lazzeri together managed to combine the Italian sty...


In [15]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.create_documents(documents)
print(f"Total chunks after splitting: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content)

Total chunks after splitting: 2641

Sample chunk:
Company: leather wear, casual jackets, artisan attention, contemporary Classic, iconic items, evolutions of menswear, menswear craftsmanship, luxury men’s wardrobe, smart casual


## Step 4: Create Embeddings (Requires OpenAI API Key)

**Option A: With OpenAI API** (skip to Option B if no API key)

In [16]:
# OPTION A: Real embeddings with OpenAI
import os

if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import OpenAIEmbeddings
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Test embedding
    test_embedding = embeddings.embed_query("luxury leather shoes")
    print(f"Embedding dimension: {len(test_embedding)}")
    print(f"Sample values: {test_embedding[:5]}")
else:
    print("No OpenAI API key found. Use Option B below for demo mode.")

Embedding dimension: 1536
Sample values: [0.0222320556640625, -0.01403045654296875, -0.0016775131225585938, 0.0030078887939453125, 0.036956787109375]


**Option B: Mock Embeddings (Demo Mode)**

In [17]:
# OPTION B: Mock embeddings for demo
from langchain_core.embeddings import Embeddings
import hashlib

class MockEmbeddings(Embeddings):
    """Mock embeddings for demo without API key"""
    
    def embed_documents(self, texts):
        return [self._hash_to_embedding(t) for t in texts]
    
    def embed_query(self, text):
        return self._hash_to_embedding(text)
    
    def _hash_to_embedding(self, text):
        """Convert text hash to deterministic embedding"""
        h = hashlib.md5(text.encode()).digest()
        return [float(b) / 255.0 for b in h[:16]]  # 16-dim vector

# Use mock or real embeddings
if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("Using OpenAI embeddings")
else:
    embeddings = MockEmbeddings()
    print("Using mock embeddings (demo mode)")

Using OpenAI embeddings


## Step 5: Create Vector Store (FAISS)

FAISS (Facebook AI Similarity Search) enables fast similarity searches.

In [18]:
try:
    from langchain_community.vectorstores import FAISS
    
    # Create vector store from chunks
    vector_store = FAISS.from_documents(chunks, embeddings)
    print(f"Vector store created with {len(chunks)} documents")
except ImportError:
    print("Note: langchain-community is deprecated.")
    print("For production: use standalone packages like langchain-faiss")
    from langchain_community.vectorstores import FAISS
    vector_store = FAISS.from_documents(chunks, embeddings)
    print(f"Vector store created with {len(chunks)} documents")

Vector store created with 2641 documents


## Step 6: Semantic Search Example

In [19]:
# Search by meaning, not keywords
query = "sustainable and eco-friendly fashion brands"

results = vector_store.similarity_search(query, k=3)

print(f"Query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content)

Query: 'sustainable and eco-friendly fashion brands'


--- Result 1 ---
Description: Sustainability  Innovation  Fashion                      Our brand pillars are sustainability, innovation and fashion. In terms of style  Ecoalf could be described as timeless, unpretentious and contemporary. When you visit our stores, speak to our sales team, touch the garments, watch our videos. There is always the same tone of voice, authentic and human. At the start of every proce...

--- Result 2 ---
fabrics and personalised production techniques to address issues of sustainability and ethical working processes. The collections are shown during Tokyo, Paris and Shanghai Fashion weeks and are available through select retailers internationally.

--- Result 3 ---
importantly to show that it is possible to design and produce beautiful high quality yet sustainable garments while having minimum impact on climate change.


In [20]:
# Try different queries
queries = [
    "luxury handmade leather goods",
    "vintage retro style clothing",
    "innovative modern design"
]

for query in queries:
    results = vector_store.similarity_search(query, k=1)
    print(f"✓ '{query}'")
    if results:
        content = results[0].page_content
        # Safely extract company name
        parts = content.split('Company:')
        if len(parts) > 1:
            company_part = parts[1].split('Description:')[0].strip()[:50]
            print(f"  → {company_part}")
        else:
            print(f"  → {content[:50]}")
    else:
        print(f"  → No results found")
    print()

✓ 'luxury handmade leather goods'
  → Description: Elegant Practical Accessories        

✓ 'vintage retro style clothing'
  → Description: Vintage&Republic, founded in 2011, is

✓ 'innovative modern design'
  → us unique today. Modern aesthetic and innovative t



## Step 7: RAG (Retrieval-Augmented Generation)

**Only available with OpenAI API key**

In [21]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

if os.getenv('OPENAI_API_KEY'):
    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
        from langchain_core.runnables import RunnablePassthrough
        
        # Setup retriever
        retriever = vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )
        
        # Format retrieved documents
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)
        
        # Custom prompt
        qa_prompt = ChatPromptTemplate.from_template("""You are an expert on Pitti Immagine fashion fair.
Using the provided exhibitor information, answer the question.
Be specific and reference company names when possible.

Context:
{context}

Question: {question}

Answer:""")
        
        # Modern LCEL chain with |
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
        
        qa_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | qa_prompt
            | llm
            | StrOutputParser()
        )
        
        print("✓ Modern LCEL retrieval chain created. Ask a question below:")
    except Exception as e:
        print(f"Error setting up retrieval chain: {type(e).__name__}")
        print("Check your OpenAI API key in .env")
else:
    print("⚠️ Skipping retrieval demo - requires OPENAI_API_KEY in .env")

✓ Modern LCEL retrieval chain created. Ask a question below:


In [22]:
# Ask questions
if os.getenv('OPENAI_API_KEY'):
    try:
        question = "Which companies specialize in leather products and when were they founded?"
        
        # Invoke LCEL chain directly with question
        answer = qa_chain.invoke(question)
        
        print(f"Question: {question}\n")
        print(f"Answer:\n{answer}")
    except Exception as e:
        print(f"Error running retrieval query: {type(e).__name__}")
        print("Verify your OPENAI_API_KEY is valid")
else:
    print("Retrieval demo requires OPENAI_API_KEY")

Question: Which companies specialize in leather products and when were they founded?

Answer:
One of the companies specializing in leather products at Pitti Immagine fashion fair is Dents. Dents is a privately owned company with 178 years of continuous tradition, known for their fine leather gloves. They were founded in 1777 and offer a range of leather wear including leather jackets, casual jackets, belts, wallets/small leather goods, handbags, luggage, and travel accessories.


## Key LangChain Concepts Demonstrated

| Concept | Example | Use Case |
|---------|---------|----------|
| **Text Splitter** | RecursiveCharacterTextSplitter | Breaking large docs into chunks for embeddings |
| **Embeddings** | OpenAIEmbeddings | Converting text to vectors for semantic search |
| **Vector Store** | FAISS | Storing and searching embeddings efficiently |
| **Retriever** | VectorStore.as_retriever() | Finding relevant documents for a query |
| **Chain** | RetrievalQA | Combining retrieval + LLM for Q&A |
| **Prompt Template** | PromptTemplate | Reusable prompt structure with variables |
| **LLM** | ChatOpenAI | Language model for text generation |

## Next Steps

1. Run `streamlit run pitti_langchain_demo.py` for interactive web UI
2. Experiment with different queries and chunk sizes
3. Try other embeddings models (Hugging Face, Cohere)
4. Implement multi-step agents
5. Add memory for conversational AI